# Proyecto de Estadística Multivariada
# Our World in Data CO2 and Greenhouse Gas Emissions dataset

## Librerías

In [51]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.decomposition import PCA




In [52]:
df_final = pd.read_csv("df_final_pca.csv")

## Análisis de Componentes Principales (PCA)

### 1. Imputar valores nulos para variables predictoras


In [53]:
df_pca = df_final.copy()

In [54]:
df_pca['population'] =  df_pca['population'].fillna(df_pca['population'].median())

In [55]:
imputer = IterativeImputer(max_iter=10, random_state=42)
predictoras = ['population', 'gdp', 'co2', 'co2_per_capita',
                    'total_ghg', 'coal_co2', 'oil_co2', 'gas_co2',
                    'cement_co2', 'co2_per_gdp', 'energy_per_capita',
                    'cumulative_co2', 'share_global_co2']
df_pca[predictoras] = imputer.fit_transform(df_pca[predictoras])


### 2. Escalar variables numéricas

In [56]:
X = df_pca[predictoras]

pca_scaler = StandardScaler()
X_scaled = pca_scaler.fit_transform(X)

### 3. Aplicar PCA

In [57]:
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

### 4. Varianza explicada

In [58]:
explained_var = pca.explained_variance_ratio_
cum_var = np.cumsum(explained_var)

for i, (ev, cv) in enumerate(zip(explained_var, cum_var), 1):
    print(f"PC{i}: Var = {ev:.3f}, Acumulada = {cv:.3f}")

PC1: Var = 0.695, Acumulada = 0.695
PC2: Var = 0.218, Acumulada = 0.912
PC3: Var = 0.045, Acumulada = 0.958
PC4: Var = 0.018, Acumulada = 0.976
PC5: Var = 0.009, Acumulada = 0.985
PC6: Var = 0.006, Acumulada = 0.991
PC7: Var = 0.004, Acumulada = 0.995
PC8: Var = 0.002, Acumulada = 0.997
PC9: Var = 0.001, Acumulada = 0.998
PC10: Var = 0.001, Acumulada = 0.999
PC11: Var = 0.001, Acumulada = 1.000
PC12: Var = 0.000, Acumulada = 1.000
PC13: Var = 0.000, Acumulada = 1.000


Vemos que PC1 y PC2 explican el 80.5% de la variabilidad de los datos y PC3 el 10.5%. Los tres juntos explican el 91.1% de la variabilidad total. A partir de PC4 cada componente aporta muy poco, por lo que no tendría sentido considerarlos.

In [59]:
A = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(predictoras))],
    index=predictoras).round(3)

A

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13
population,0.284,-0.031,-0.550,0.566,0.080,-0.294,0.297,0.019,0.031,-0.182,-0.282,0.026,-0.005
gdp,0.321,0.000,0.047,-0.245,0.483,0.285,0.634,-0.116,0.174,0.091,0.213,0.147,-0.001
co2,0.330,-0.001,-0.015,-0.186,-0.188,-0.124,-0.102,0.062,-0.041,-0.039,0.004,0.391,0.797
co2_per_capita,0.011,0.588,0.062,0.052,-0.244,0.084,0.060,0.220,0.566,-0.416,0.174,-0.067,-0.002
total_ghg,0.323,-0.019,-0.148,0.206,0.090,0.492,-0.257,0.635,-0.127,0.237,0.135,-0.144,0.002
coal_co2,0.326,-0.007,-0.094,-0.273,-0.287,-0.228,-0.064,0.168,-0.031,0.061,0.081,0.536,-0.587
oil_co2,0.324,0.018,0.121,0.181,0.081,0.463,-0.398,-0.510,0.028,-0.303,-0.241,0.206,-0.123
gas_co2,0.272,0.025,0.694,0.315,0.289,-0.411,-0.037,0.180,-0.157,-0.143,0.106,-0.022,-0.045
cement_co2,0.322,-0.008,-0.019,-0.484,-0.101,-0.007,0.118,0.101,-0.317,-0.419,-0.310,-0.502,-0.054
co2_per_gdp,-0.019,0.560,-0.277,-0.242,0.570,-0.252,-0.365,-0.039,-0.073,0.128,-0.077,0.005,-0.001


A partir de la tabla podemos interpretar lo siguiente:

Vemos que el primer componente principal explica el 59.3% de la variabilidad total. Las variables más relacionadas con PC1 son: co2 → 0.357; total_ghg → 0.351; oil_co2, coal_co2, gas_co2 → ~0.30; gdp → 0.342 y share_global_co2 → 0.322. Estos miden el tamaño absoluto del impacto climático de un país, es decir, el volumen total.

El segundo componente explica el 21.3% de la variabilidad, está asociado con: co2_per_capita → 0.595; energy_per_capita → 0.570 y co2_per_gdp → 0.559. Este componente nos dice que tan "intensivo" es en el uso de la energía y en la generación de CO2 independientemente de su tamaño. En otras palabras, PC2 nos ayuda a diferenciar entre países que tienen una gran huella individual.

PC3 asociado a: cement_co2 → 0.495; coal_co2 → 0.340; population → 0.424; gas_co2 → –0.470; oil_co2 → –0.316 y cumulative_co2 → –0.340. Explica el 10.5% de la variabilidad de los datos. Estos explican de qué fuentes proviene el CO2:

- Valores positivos → economías dependientes de cemento y carbón
- Valores negativos → economías más dependientes de gas y petróleo

Podemos interpretarlos así:
1.  Escala de emisiones y actividad económica (PC1)
2.  Intensidad de carbono y consumo energético per cápita (PC2)
3.  Composición de matriz energética (PC3)

Estos componentes se utilizarán posteriormente para PCA.